Ecommerce Project: Planning: 
The project represents the data collected during the landing page optimization

In [ ]:
import pandas as pd

In [46]:
df = pd.read_csv(r"C:\Users\pgoyal\Downloads\hypothesistesting_learning\Datasets\ab_test.csv")
#df = pd.read_csv(r".\hypothesistesting_learning\Datasets\ab_test.csv")

In [47]:
df.head(5)

,id,time,con_treat,page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   id         294478 non-null  int64 
 1   time       294478 non-null  object
 2   con_treat  294478 non-null  object
 3   page       294478 non-null  object
 4   converted  294478 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


Analysis Phase: EXPLORATORY DATA ANALYSIS

In [49]:
df.shape

(294478, 5)

In [50]:
df.columns = ["user_id", "timestamp", "group", "landing_page", "converted"]
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


In [51]:
#numer of rows and unique users
print(f'Number of rows: {df.shape[0]}')
print(f'Number of unique users: {df.user_id.nunique()}')

Number of rows: 294478
Number of unique users: 290584


In [52]:
#missing values
df.isna().sum()

user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64

In [53]:
#Does the number of new_page and treatment match?
n_treat = df[df["group"] == "treatment"].shape[0]
n_new_page = df[df["landing_page"] == "new_page"].shape[0]
difference = n_treat - n_new_page

pd.DataFrame({
    'N treatment': [n_treat],
    'N new_page': [n_new_page],
    'Difference': [difference]
})

,N treatment,N new_page,Difference
0,147276,147239,37


It does show that Treatment page and new page are not the same. Most likely, there are users who are treatment but are still seeing old urls.

In [54]:
df[(df['group'] == "treatment") & (df['landing_page'] == "old_page")]
#df[(df["group"] == "treatment") & (df["landing_page"] == "old_page")]

,user_id,timestamp,group,landing_page,converted
308,857184,34:59.8,treatment,old_page,0
327,686623,26:40.7,treatment,old_page,0
357,856078,29:30.4,treatment,old_page,0
685,666385,11:54.8,treatment,old_page,0
713,748761,47:44.4,treatment,old_page,0
...,...,...,...,...,...
293773,688144,34:50.5,treatment,old_page,1
293817,876037,15:09.0,treatment,old_page,1
293917,738357,37:55.7,treatment,old_page,0
294014,813406,25:33.2,treatment,old_page,0


There are 1965 rows of data with treatment group receiving old page. This needs to be removed from the dataset.

In [55]:
df_mismatch = df[(df['group'] == "treatment") & (df['landing_page'] == "old_page")
               |(df['group'] == "control") & (df['landing_page'] == "new_page")]

n_mismatch = df_mismatch.shape[0]

percent_mismatch = round(n_mismatch / len(df) * 100, 2)
print(f'Number of mismatched rows: {n_mismatch} rows')
print(f'Percent of mismatched rows: {percent_mismatch} percent')

Number of mismatched rows: 3893 rows
Percent of mismatched rows: 1.32 percent


Since we cannot be sure of the output for the mismatched rows, for the purpose of the excercise, remove the 

In [56]:
df2 = df[(df['group'] == "treatment") & (df['landing_page'] == "new_page")
        |(df['group'] == "control") & (df['landing_page'] == "old_page")]

len(df2)

290585

In [57]:
# Double Check all of the correct rows were removed - this should be 0
df2[((df2['group'] == 'treatment') == (df2['landing_page'] == 'new_page')) == False].shape[0]

0

In [58]:
# Another way to double Check all of the correct rows were removed 
df_mismatch = df2[(df2["group"] == "treatment") & (df2["landing_page"] == "old_page")
               |(df2["group"] == "control") & (df2["landing_page"] == "new_page")]

n_mismatch = df_mismatch.shape[0]

percent_mismatch = round(n_mismatch / len(df2) * 100, 2)
print(f'Number of mismatched rows: {n_mismatch} rows')
print(f'Percent of mismatched rows: {percent_mismatch} percent')

Number of mismatched rows: 0 rows
Percent of mismatched rows: 0.0 percent


In [59]:
# unique user id in df2 
df2.user_id.nunique()

290584

In [60]:
# number of repeated ids in df2
len(df2) - df2.user_id.nunique()

1

In [61]:
# Display the duplicated row 
df2[df2.duplicated("user_id") == True]

,user_id,timestamp,group,landing_page,converted
2893,773192,55:59.6,treatment,new_page,0


In [62]:
#drop the duplicated row
df2 = df2.drop_duplicates("user_id")

In [63]:
# Douple Check that it is actually dropped
len(df2) - df2.user_id.nunique()

0

EDA Complete

Construct Phase: Couple of metrics to look at: 
1) Mean conversion rate on the entire sample provided
2) Conversion rate by control group
3) Conversion by user_id i.e determine how is the variation across the users.
4) Hypothesis Testing to determine if the new variant is better than the old variant

In [66]:
# Mean Conversion rate
mean_conversion = df2['converted'].mean()
print(f'Mean Conversion:{mean_conversion}')

Mean Conversion:0.11959708724499628
